In [1]:
import torch, torchaudio, random, itertools
import torch.nn as nn
import torch.optim as optim
import numpy as np, pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE, HOP_SECONDS = 44100, 0.010
HOP, N_FFT, N_MELS = int(HOP_SECONDS * SAMPLE_RATE), 2048, 128

class OnsetCRNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.mel   = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        self.rnn = nn.GRU(128 * (N_MELS // 4), 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head, self.onset_head, self.offset_head = nn.Linear(256, 2), nn.Linear(256, 1), nn.Linear(256, 1)
    def forward(self, x):
        s = self.cnn(self.to_db(self.mel(x)))
        B, C, M, T = s.shape
        s, _ = self.rnn(s.reshape(B, C*M, T).transpose(1, 2))
        s = self.drop(s)
        return self.state_head(s), self.onset_head(s).squeeze(-1), self.offset_head(s).squeeze(-1)

weight_files = sorted(Path("/kaggle/input").rglob("biggerCRNN_ext4_seed*.pt"))

# ---- find the CSD upload folder (the one holding index.csv + audio/ + annot/) ----
CSD = next(p.parent for p in Path("/kaggle/input").rglob("index.csv")
           if (p.parent / "audio").exists() and (p.parent / "annot").exists())
index = pd.read_csv(CSD / "index.csv")
index["wav"]   = index.file.apply(lambda f: CSD / "audio" / f"{f}.wav")
index["label"] = index.file.apply(lambda f: CSD / "annot" / f"{f}.csv")

# ---- checks ----
print("device :", device, "| weights:", [w.name for w in weight_files], "(expect 3)")
print("CSD folder:", CSD)
print("files:", len(index), "(expect 109) | songs:", index.song.nunique(), "(expect 82)")
print("all wavs found:", index.wav.apply(Path.exists).all(), "| all labels found:", index.label.apply(Path.exists).all())
print(index.groupby("lang").agg(files=("file", "size"), songs=("song", "nunique"), notes=("notes", "sum")))

device : cuda | weights: ['biggerCRNN_ext4_seed0.pt', 'biggerCRNN_ext4_seed1.pt', 'biggerCRNN_ext4_seed2.pt'] (expect 3)
CSD folder: /kaggle/input/datasets/alirahmatpour/vocalset-onset/csd_upload
files: 109 (expect 109) | songs: 82 (expect 82)
all wavs found: True | all labels found: True
         files  songs  notes
lang                        
english     59     32   8846
korean      50     50   7897


In [2]:
songs = []
for r in index.itertuples():
    wf, sr = torchaudio.load(str(r.wav))
    wave = wf.mean(0)                                        # stereo -> mono
    if sr != SAMPLE_RATE:
        wave = torchaudio.functional.resample(wave, sr, SAMPLE_RATE)
    lab = pd.read_csv(r.label)
    songs.append({"key": r.file, "song": r.song, "lang": r.lang, "sr": sr, "wave": wave,
                  "on": lab["start"].values, "off": lab["end"].values})
print("songs:", len(songs), "| sample rates:", sorted({s['sr'] for s in songs}),
      "| langs:", pd.Series([s['lang'] for s in songs]).value_counts().to_dict())
print("notes:", sum(len(s['on']) for s in songs))

def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i - peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def f1_sets(pred_list, true_list, tol):
    tp = fp = fn = 0
    for pred, t_idx in zip(pred_list, true_list):
        matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi - ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(set(pred) - matched)
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

frames = [set((s["on"] / HOP_SECONDS).astype(int)) for s in songs]

print("\n=== HUMAN CEILING ===")
print("not available: CSD has one clean label set per file")

rows, offsets = [], []
for w in weight_files:
    model = OnsetCRNN().to(device)
    model.load_state_dict(torch.load(w, map_location=device)); model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(s["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for s in songs]
    peaks08 = [pick_peaks(p, 0.8) for p in probs]
    best_thr = max([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
                   key=lambda th: f1_sets([pick_peaks(p, th) for p in probs], frames, 2))
    peaksB = [pick_peaks(p, best_thr) for p in probs]
    row = {"seed": w.stem[-1],
           "F1_20": f1_sets(peaks08, frames, 2), "F1_50": f1_sets(peaks08, frames, 5),
           "best_thr": best_thr, "bestF1_20": f1_sets(peaksB, frames, 2),
           "bestF1_50": f1_sets(peaksB, frames, 5)}
    for lang in ["english", "korean"]:                      # per-language, ±50 ms
        idx = [i for i, s in enumerate(songs) if s["lang"] == lang]
        row[f"{lang}_F1_50"] = f1_sets([peaks08[i] for i in idx], [frames[i] for i in idx], 5)
    rows.append(row)
    for pred, t in zip(peaks08, frames):
        t = np.array(sorted(t))
        if len(t) == 0: continue
        for pi in pred:
            dd = pi - t[np.argmin(np.abs(t - pi))]
            if abs(dd) <= 20: offsets.append(dd * 10)
    del model; torch.cuda.empty_cache()

zs = pd.DataFrame(rows)
print("\n=== ZERO-SHOT: VocalSet model on CSD (mean ± std over 3 seeds) ===")
for c in ["F1_20", "F1_50", "bestF1_20", "bestF1_50", "english_F1_50", "korean_F1_50"]:
    print(f"{c:14s}: {zs[c].mean():.3f} ± {zs[c].std(ddof=1):.3f}")
print("best thresholds picked:", sorted(zs.best_thr.unique()))
off_arr = np.array(offsets)
print(f"\ntiming offset (model - human): median {np.median(off_arr):+.0f} ms | within ±20ms: {np.mean(np.abs(off_arr)<=20):.0%}")

songs: 109 | sample rates: [44100] | langs: {'english': 59, 'korean': 50}
notes: 16743

=== HUMAN CEILING ===
not available: CSD has one clean label set per file

=== ZERO-SHOT: VocalSet model on CSD (mean ± std over 3 seeds) ===
F1_20         : 0.514 ± 0.034
F1_50         : 0.752 ± 0.022
bestF1_20     : 0.517 ± 0.031
bestF1_50     : 0.755 ± 0.019
english_F1_50 : 0.648 ± 0.028
korean_F1_50  : 0.868 ± 0.015
best thresholds picked: [np.float64(0.7), np.float64(0.8)]

timing offset (model - human): median +10 ms | within ±20ms: 55%


In [5]:
# ---- split by SONG (a+b keys stay together), about 1/3 of songs to test, balanced by language ----
rng = random.Random(0)
test_songs = set()
for lang in ["english", "korean"]:
    ss = sorted({s["song"] for s in songs if s["lang"] == lang}); rng.shuffle(ss)
    test_songs |= set(ss[: len(ss) // 3])
test_i = [i for i, s in enumerate(songs) if s["song"] in test_songs]
pool_i = [i for i, s in enumerate(songs) if s["song"] not in test_songs]
rng.shuffle(pool_i)                                            # first N = the N training files
print("test songs:", len(test_songs), "| test files:", len(test_i), "| training pool:", len(pool_i))
print("test langs:", pd.Series([songs[i]["lang"] for i in test_i]).value_counts().to_dict())
print("song overlap train/test:", len({songs[i]['song'] for i in test_i} & {songs[i]['song'] for i in pool_i}), "(must be 0)")

SOFT_WIDTH = 2
def make_targets(s):
    nfr = 1 + s["wave"].shape[0] // HOP
    st = np.zeros(nfr, np.int64); on_t = np.zeros(nfr, np.float32); off_t = np.zeros(nfr, np.float32)
    def bump(arr, c):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH + 1):
            j = c + d
            if 0 <= j < nfr: arr[j] = max(arr[j], np.exp(-(d*d) / (2 * (SOFT_WIDTH/2)**2)))
    for a, b in zip(s["on"], s["off"]):
        fa, fb = int(a / HOP_SECONDS), int(b / HOP_SECONDS)
        st[fa:fb] = 1; bump(on_t, fa); bump(off_t, fb)
    return torch.tensor(st), torch.tensor(on_t), torch.tensor(off_t)
tg = [make_targets(s) for s in songs]

def train_model(start, train_idx, seed, epochs, lr, crop=1500):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = OnsetCRNN().to(device)
    if start is not None:
        model.load_state_dict(torch.load(start, map_location=device))
    if not train_idx: return model
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    ce, bce = nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0, device=device))
    for ep in range(epochs):
        model.train()
        for i in random.sample(train_idx, len(train_idx)):
            st, on, off = tg[i]
            nfr = len(st); a = np.random.randint(0, max(1, nfr - crop)); b = min(nfr, a + crop)
            ps, po, pf = model(songs[i]["wave"][a*HOP : b*HOP].view(1, 1, -1).to(device))
            F = min(ps.shape[1], b - a)
            loss = (ce(ps[0, :F], st[a:a+F].to(device)) + bce(po[0, :F], on[a:a+F].to(device))
                    + bce(pf[0, :F], off[a:a+F].to(device)))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(songs[i]["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for i in test_i]
    t = [frames[i] for i in test_i]
    out = {}
    for thr_name, thr in [("", 0.8), ("best_", None)]:
        if thr is None:
            thr = max([0.3,0.4,0.5,0.6,0.7,0.8,0.9],
                      key=lambda th: f1_sets([pick_peaks(p, th) for p in probs], t, 5))
        peaks = [pick_peaks(p, thr) for p in probs]
        for tol_name, tol in [("20", 2), ("50", 5)]:
            out[f"{thr_name}F1_{tol_name}"] = f1_sets(peaks, t, tol)
    return out

# ---- SAME training for both: 40 rounds, same step size (fair comparison) ----
SETUPS = {"pretrained": dict(epochs=40, lr=1e-3),     # starts from VocalSet weights
          "scratch":    dict(epochs=40, lr=1e-3)}     # starts empty
N_LIST = sorted({0, 3, 9, 18, len(pool_i)})
rows = []
for N in N_LIST:
    for k, w in enumerate(weight_files):
        for name, cfg in SETUPS.items():
            if N == 0 and name == "scratch": continue
            model = train_model(w if name == "pretrained" else None, pool_i[:N], seed=k, **cfg)
            r = evaluate(model); r.update({"N": N, "start": name, "seed": k}); rows.append(r)
            print(f"N={N:2d} | {name:10s} | seed {k} | ±20 {r['F1_20']:.3f} | ±50 {r['F1_50']:.3f}"
                  f" | best thr ±50 {r['best_F1_50']:.3f}")
            del model; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/onset_V3_CSD_zeroshot_vs_finetune.csv", index=False)
print("\n=== CSD test songs — mean ± std over 3 seeds (both: 40 rounds, lr 1e-3) ===")
for (N, start), g in res.groupby(["N", "start"]):
    print(f"N={N:2d} {start:10s} | ±20 {g.F1_20.mean():.3f} ± {g.F1_20.std(ddof=1):.3f}"
          f" | ±50 {g.F1_50.mean():.3f} ± {g.F1_50.std(ddof=1):.3f} | best-thr ±50 {g.best_F1_50.mean():.3f}")

test songs: 26 | test files: 34 | training pool: 75
test langs: {'english': 18, 'korean': 16}
song overlap train/test: 0 (must be 0)
N= 0 | pretrained | seed 0 | ±20 0.500 | ±50 0.755 | best thr ±50 0.760
N= 0 | pretrained | seed 1 | ±20 0.538 | ±50 0.764 | best thr ±50 0.770
N= 0 | pretrained | seed 2 | ±20 0.570 | ±50 0.792 | best thr ±50 0.792
N= 3 | pretrained | seed 0 | ±20 0.740 | ±50 0.879 | best thr ±50 0.879
N= 3 | scratch    | seed 0 | ±20 0.738 | ±50 0.862 | best thr ±50 0.863
N= 3 | pretrained | seed 1 | ±20 0.732 | ±50 0.867 | best thr ±50 0.868
N= 3 | scratch    | seed 1 | ±20 0.666 | ±50 0.824 | best thr ±50 0.824
N= 3 | pretrained | seed 2 | ±20 0.700 | ±50 0.854 | best thr ±50 0.859
N= 3 | scratch    | seed 2 | ±20 0.726 | ±50 0.855 | best thr ±50 0.856
N= 9 | pretrained | seed 0 | ±20 0.760 | ±50 0.896 | best thr ±50 0.896
N= 9 | scratch    | seed 0 | ±20 0.746 | ±50 0.881 | best thr ±50 0.881
N= 9 | pretrained | seed 1 | ±20 0.704 | ±50 0.890 | best thr ±50 0.890
N= 